In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
"""
OBO Ontology Graph Parser adapted for Google Colab.

This script parses OBO files and extracts the ontology graph structure,
focusing on 'is_a' relationships to build parent-child hierarchies.

Usage in Colab:
1. Upload your OBO file (e.g., 'cl.obo') to your Colab environment.
   You can do this by clicking the folder icon on the left sidebar ->
   'Files' tab -> 'Upload to session storage'.
2. Run the `main` function with your input OBO file name and desired output JSON file name.
   Example: main('cl.obo', 'cl_graph.json')
"""

import re
import json
from pathlib import Path
from typing import Dict, List, Set


class OBOGraphParser:
    """Parser for OBO ontology graph structure."""

    def __init__(self):
        self.terms: Dict[str, Dict] = {}
        self.graph: Dict[str, List[str]] = {}  # term_id -> [parent_ids]
        self.stats = {
            'total_terms': 0,
            'terms_with_parents': 0,
            'total_relationships': 0
        }

    def parse_file(self, filepath: str) -> None:
        """Parse an OBO file and extract graph structure."""
        print(f"Processing {filepath}...")

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
        except FileNotFoundError:
            print(f"Error: File not found at {filepath}. Please ensure the file is uploaded or the path is correct.")
            return
        except Exception as e:
            print(f"Error reading file {filepath}: {e}")
            return

        # Split into term blocks
        blocks = self._split_into_blocks(content)

        for block in blocks:
            if not block.strip():
                continue

            lines = [line.strip() for line in block.strip().split('\n')]
            if not lines:
                continue

            # Check if the block starts with a valid block type, e.g., [Term]
            if not lines[0].startswith('[') or not lines[0].endswith(']'):
                continue # Skip malformed blocks

            block_type = lines[0].strip('[]')

            if block_type == 'Term':
                self._parse_term_block(lines[1:])

        print(f"✓ Successfully processed {filepath}")

    def _split_into_blocks(self, content: str) -> List[str]:
        """Split OBO content into blocks based on [Term], [Typedef], etc."""
        # This regex looks for lines starting with '[', followed by one or more
        # characters that are not ']', and ending with ']', followed by a newline.
        # It then splits the content based on these block headers.
        block_pattern = re.compile(r'^\[[^\]]+\]\n', re.MULTILINE)

        # Find all block headers and their positions
        matches = list(block_pattern.finditer(content))

        if not matches:
            # If no explicit blocks are found, treat the whole content as one block
            return [content]

        blocks = []
        for i, match in enumerate(matches):
            start = match.start()
            # The end of the current block is the start of the next block,
            # or the end of the content if it's the last block.
            end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
            blocks.append(content[start:end])

        return blocks

    def _parse_term_block(self, lines: List[str]) -> None:
        """Parse a [Term] block and extract term info and relationships."""
        term_id = None
        name = ""
        parents = []
        is_obsolete = False

        for line in lines:
            if not line or line.startswith('!'):  # Skip empty lines or comments
                continue

            if ':' not in line:  # Skip lines not containing key-value pairs
                continue

            key, value = line.split(':', 1)
            key = key.strip()
            value = value.strip()

            if key == 'id':
                # Only process CL (Cell Ontology) terms
                if value.startswith('CL:'):
                    term_id = value
            elif key == 'name':
                name = value
            elif key == 'is_a':
                # Extract parent ID from is_a relationship
                # Format examples:
                # "is_a: CL:0000021 {is_inferred="true"} ! female germ cell"
                # "is_a: CL:0000021 ! female germ cell"
                # "is_a: CL:0000021"
                parent_match = re.match(r'([A-Z_]+:\d+)', value)  # Updated regex to include underscore
                if parent_match:
                    parent_id = parent_match.group(1)
                    # Only include CL (Cell Ontology) terms
                    if parent_id.startswith('CL:'):
                        parents.append(parent_id)
            elif key == 'is_obsolete':
                is_obsolete = value.lower() == 'true'

        # Store term info if we have a valid CL ID and it's not obsolete
        if term_id and term_id.startswith('CL:') and not is_obsolete:
            self.terms[term_id] = {
                'name': name,
                'parents': parents
            }

            # Store in graph structure
            self.graph[term_id] = parents

            self.stats['total_terms'] += 1
            if parents:
                self.stats['terms_with_parents'] += 1
                self.stats['total_relationships'] += len(parents) == 'true'

        # Store term info if we have a valid ID and it's not obsolete
        if term_id and not is_obsolete:
            self.terms[term_id] = {
                'name': name,
                'parents': parents
            }

            # Store in graph structure
            self.graph[term_id] = parents

            self.stats['total_terms'] += 1
            if parents:
                self.stats['terms_with_parents'] += 1
                self.stats['total_relationships'] += len(parents)

    def save_graph(self, output_file: str) -> None:
        """Save the ontology graph to a JSON file."""
        graph_data = {
            'metadata': {
                'total_terms': self.stats['total_terms'],
                'terms_with_parents': self.stats['terms_with_parents'],
                'total_relationships': self.stats['total_relationships']
            },
            'terms': self.terms,
            'graph': self.graph
        }

        try:
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(graph_data, f, indent=2, ensure_ascii=False)

            print(f"\n✓ Successfully saved ontology graph to {output_file}")
            self._print_stats()

        except Exception as e:
            print(f"✗ Error saving graph file: {e}")

    def _print_stats(self) -> None:
        """Print processing statistics."""
        print(f"\nProcessing Statistics:")
        print(f"  Total terms: {self.stats['total_terms']}")
        print(f"  Terms with parents: {self.stats['terms_with_parents']}")
        print(f"  Total is_a relationships: {self.stats['total_relationships']}")

    def print_sample_hierarchy(self, n: int = 5) -> None:
        """Print a sample of the hierarchy structure."""
        print(f"\nSample hierarchy (first {n} terms):")
        count = 0
        for term_id, term_data in self.terms.items():
            if count >= n:
                break

            print(f"  {term_id}: {term_data['name']}")
            if term_data['parents']:
                for parent_id in term_data['parents']:
                    parent_name = self.terms.get(parent_id, {}).get('name', 'Unknown')
                    print(f"    ↳ is_a: {parent_id} ({parent_name})")
            else:
                print(f"    ↳ (root term or no 'is_a' parents found)")
            print()
            count += 1

    def find_roots(self) -> List[str]:
        """Find root terms (terms with no 'is_a' parents)."""
        roots = []
        for term_id, parents in self.graph.items():
            if not parents: # A term is a root if it has no 'is_a' parents in the parsed graph
                roots.append(term_id)
        return roots

    def find_leaves(self) -> List[str]:
        """Find leaf terms (terms that are not 'is_a' parents of any other term)."""
        all_parents_in_graph = set()
        for parents_list in self.graph.values():
            all_parents_in_graph.update(parents_list)

        leaves = []
        for term_id in self.graph.keys():
            # A term is a leaf if it exists in the graph but is not listed as a parent for any other term
            if term_id not in all_parents_in_graph:
                leaves.append(term_id)
        return leaves


def main(obo_file_path: str, output_json_path: str, sample_terms_to_display: int = 50):
    """
    Main function to parse an OBO file and save its graph structure.
    Designed for use in Google Colab.

    Args:
        obo_file_path (str): The path to the input OBO file.
        output_json_path (str): The desired path for the output JSON file.
        sample_terms_to_display (int): Number of sample terms to display in the output.
    """
    try:
        parser_obj = OBOGraphParser()

        # Process the input file
        parser_obj.parse_file(obo_file_path)

        if not parser_obj.terms:
            print("No terms found in the input file or parsing failed.")
            return

        # Show sample hierarchy
        parser_obj.print_sample_hierarchy(sample_terms_to_display)

        # Find and display roots and leaves
        roots = parser_obj.find_roots()
        leaves = parser_obj.find_leaves()

        print(f"\nGraph Analysis:")
        print(f"  Root terms (no 'is_a' parents in the parsed graph): {len(roots)}")
        print(f"  Leaf terms (no 'is_a' children in the parsed graph): {len(leaves)}")

        # Save the graph
        parser_obj.save_graph(output_json_path)

        print(f"\n✓ Graph parsing complete! Output saved to: {output_json_path}")

        # Instructions for downloading the file in Colab
        print(f"\nTo download the output file '{output_json_path}' in Colab, run the following in a new cell:")
        print(f"  from google.colab import files")
        print(f"  files.download('{output_json_path}')")

    except Exception as e:
        print(f"✗ An unexpected error occurred: {e}")


In [13]:
main('cl.obo', 'cl_graph.json', sample_terms_to_display=10)


Processing cl.obo...
✓ Successfully processed cl.obo

Sample hierarchy (first 10 terms):
  CL:0000000: cell
    ↳ (root term or no 'is_a' parents found)

  CL:0000001: primary cultured cell
    ↳ is_a: CL:0000010 (cultured cell)

  CL:0000005: neural crest derived fibroblast
    ↳ is_a: CL:0000057 (fibroblast)

  CL:0000006: neuronal receptor cell
    ↳ is_a: CL:0000101 (sensory neuron)
    ↳ is_a: CL:0000197 (sensory receptor cell)

  CL:0000007: early embryonic cell (metazoa)
    ↳ is_a: CL:0002321 (embryonic cell (metazoa))

  CL:0000008: migratory cranial neural crest cell
    ↳ is_a: CL:0000333 (migratory neural crest cell)

  CL:0000010: cultured cell
    ↳ is_a: CL:0000578 (experimentally modified cell in vitro)

  CL:0000011: migratory trunk neural crest cell
    ↳ is_a: CL:0000333 (migratory neural crest cell)

  CL:0000014: germ line stem cell
    ↳ is_a: CL:0000034 (stem cell)
    ↳ is_a: CL:0000039 (germ line cell)

  CL:0000015: male germ cell
    ↳ is_a: CL:0000586 (germ 

In [15]:
"""
NetworkX Graph Analyzer adapted for Google Colab.

This script reads a JSON graph generated by the OBO Ontology Graph Parser
and analyzes the hierarchy depths using NetworkX.

Usage in Colab:
1. Upload your JSON graph file (e.g., 'cl_graph.json') to your Colab environment.
   You can do this by clicking the folder icon on the left sidebar ->
   'Files' tab -> 'Upload to session storage'.
2. Run the `main_analyzer` function with your input JSON file name and desired
   number of sample terms to display.
   Example: main_analyzer('cl_graph.json', sample_terms_to_display=10)
"""

import json
import networkx as nx
from collections import defaultdict
from typing import Dict, List


def load_graph(json_file: str) -> nx.DiGraph:
    """Load the JSON graph and create a NetworkX directed graph."""
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: JSON graph file not found at {json_file}.")
        raise
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {json_file}. Is it a valid JSON file?")
        raise
    except Exception as e:
        print(f"An unexpected error occurred while loading {json_file}: {e}")
        raise

    G = nx.DiGraph()

    # Add nodes with names
    if 'terms' not in data:
        print("Warning: 'terms' key not found in JSON data. Nodes will be added without names.")
        terms_data = {}
    else:
        terms_data = data['terms']

    for term_id, term_data in terms_data.items():
        G.add_node(term_id, name=term_data.get('name', 'Unknown'))

    # Add edges (parent -> child relationships)
    if 'graph' not in data:
        print("Warning: 'graph' key not found in JSON data. No edges will be added.")
    else:
        for term_id, parents in data['graph'].items():
            for parent_id in parents:
                # Only add if parent exists in our data and is not the same as the child
                if parent_id in G and parent_id != term_id:
                    G.add_edge(parent_id, term_id)
                elif parent_id not in G:
                    # Optionally add parent as a node even if not in 'terms' section,
                    # if it's referenced as a parent. This can happen if the parser
                    # was configured to only include certain types of terms.
                    G.add_node(parent_id, name='(Referenced Parent)')

    return G


def calculate_depths(G: nx.DiGraph) -> Dict[str, int]:
    """
    Calculate depth of each node from root nodes.
    Depth is defined as the shortest path length from any root node to the current node.
    A root node has a depth of 0.
    """
    depths = {}

    # Find root nodes (nodes with no incoming edges in the current graph)
    roots = [node for node in G.nodes() if G.in_degree(node) == 0]
    print(f"Found {len(roots)} root nodes in the graph.")

    # Initialize depths for all nodes to infinity
    for node in G.nodes():
        depths[node] = float('inf')

    # Set depth of root nodes to 0
    for root in roots:
        depths[root] = 0

    # Use a breadth-first search (BFS) like approach to calculate depths
    # This is more efficient than calculating shortest paths for each node individually
    # from all roots, especially for large graphs.
    q = [(root, 0) for root in roots] # Queue of (node, current_depth)
    visited = set(roots)

    head = 0
    while head < len(q):
        current_node, current_depth = q[head]
        head += 1

        # Update depth if a shorter path is found (shouldn't happen with BFS on unweighted graph)
        if current_depth < depths[current_node]:
            depths[current_node] = current_depth

        # Explore neighbors
        for neighbor in G.successors(current_node):
            if neighbor not in visited:
                visited.add(neighbor)
                depths[neighbor] = current_depth + 1
                q.append((neighbor, current_depth + 1))
            elif current_depth + 1 < depths[neighbor]:
                # If a shorter path is found to an already visited node (e.g., in a DAG)
                depths[neighbor] = current_depth + 1
                q.append((neighbor, current_depth + 1)) # Re-add to queue to re-evaluate its descendants

    # Handle nodes unreachable from any identified root (they might be part of disconnected components
    # or have incoming edges from nodes not included in the parsed 'terms'/'graph' structure).
    # Assign them a depth of 0 or a special value, depending on interpretation.
    # Here, we'll assign 0, treating them as effective roots for their sub-components.
    for node in G.nodes():
        if depths[node] == float('inf'):
            depths[node] = 0 # Or a different indicator like -1, depending on desired behavior
            # print(f"Warning: Node {node} is unreachable from any identified root; assigned depth 0.")

    return depths


def print_depth_histogram(depths: Dict[str, int], total_terms_in_original_data: int):
    """Print histogram of depths."""
    depth_counts = defaultdict(int)
    for depth in depths.values():
        depth_counts[depth] += 1

    print("\nDepth Histogram:")
    print("Depth | Count | Percentage")
    print("-" * 30)

    # Use the total terms from the parsed OBO data (if available) for percentage,
    # otherwise use the number of nodes in the graph.
    total_nodes_in_graph = len(depths)

    for depth in sorted(depth_counts.keys()):
        count = depth_counts[depth]
        percentage = (count / total_nodes_in_graph) * 100
        print(f"{depth:5d} | {count:5d} | {percentage:6.1f}%")

    print(f"\nTotal nodes in graph: {total_nodes_in_graph}")
    if depths: # Avoid error if depths is empty
        print(f"Max depth: {max(depths.values())}")
        print(f"Average depth: {sum(depths.values()) / total_nodes_in_graph:.2f}")
    else:
        print("No depths to display.")


def print_sample_depths(depths: Dict[str, int], terms_data: Dict[str, Dict], n: int = 10):
    """Print sample terms with their depths."""
    print(f"\nSample terms with depths (first {n} by depth, then ID):")

    # Sort by depth, then by ID for consistent output
    sorted_terms = sorted(depths.items(), key=lambda x: (x[1], x[0]))

    for i, (term_id, depth) in enumerate(sorted_terms[:n]):
        name = terms_data.get(term_id, {}).get('name', 'Unknown')
        print(f"  Depth {depth}: {term_id} - {name}")


def main_analyzer(json_file_path: str, sample_terms_to_display: int = 10):
    """
    Main function to analyze an ontology graph JSON file.
    Designed for use in Google Colab.

    Args:
        json_file_path (str): The path to the input JSON graph file.
        sample_terms_to_display (int): Number of sample terms to display.
    """
    try:
        print(f"Loading graph from {json_file_path}...")

        # Load the graph
        G = load_graph(json_file_path)
        print(f"Loaded graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

        # Load terms data separately for names (as load_graph might add nodes not in original 'terms')
        with open(json_file_path, 'r', encoding='utf-8') as f:
            full_data = json.load(f)
        terms_data = full_data.get('terms', {})

        # Calculate depths
        print("Calculating depths...")
        depths = calculate_depths(G)

        # Print results
        print_depth_histogram(depths, len(terms_data))
        print_sample_depths(depths, terms_data, sample_terms_to_display)

        # Optional: Save depths to file
        output_file = json_file_path.replace('.json', '_depths.json')
        depth_output_data = {
            term_id: {
                'name': terms_data.get(term_id, {}).get('name', '(Unknown Name)'),
                'depth': depth
            }
            for term_id, depth in depths.items()
        }

        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(depth_output_data, f, indent=2, ensure_ascii=False)

        print(f"\n✓ Depths saved to {output_file}")

        # Instructions for downloading the file in Colab
        print(f"\nTo download the output file '{output_file}' in Colab, run the following in a new cell:")
        print(f"  from google.colab import files")
        print(f"  files.download('{output_file}')")

    except Exception as e:
        print(f"✗ An unexpected error occurred during analysis: {e}")



In [17]:
main_analyzer('cl_graph.json', sample_terms_to_display=500)


Loading graph from cl_graph.json...
Loaded graph with 2922 nodes and 4113 edges.
Calculating depths...
Found 11 root nodes in the graph.

Depth Histogram:
Depth | Count | Percentage
------------------------------
    0 |    11 |    0.4%
    1 |    32 |    1.1%
    2 |   185 |    6.3%
    3 |   515 |   17.6%
    4 |   603 |   20.6%
    5 |   631 |   21.6%
    6 |   472 |   16.2%
    7 |   194 |    6.6%
    8 |   117 |    4.0%
    9 |   103 |    3.5%
   10 |    47 |    1.6%
   11 |    11 |    0.4%
   12 |     1 |    0.0%

Total nodes in graph: 2922
Max depth: 12
Average depth: 4.85

Sample terms with depths (first 500 by depth, then ID):
  Depth 0: CL:0000000 - cell
  Depth 0: CL:0017500 - neutrophillic cytoplasm
  Depth 0: CL:0017502 - acidophilic cytoplasm
  Depth 0: CL:0017503 - basophilic cytoplasm
  Depth 0: CL:0017504 - polychromatophilic cytoplasm
  Depth 0: CL:0017506 - banded nucleus
  Depth 0: CL:0017507 - reniform nucleus
  Depth 0: CL:0017508 - cartwheel heterochromatin
  Dep

In [26]:
import json
from datasets import load_dataset, load_from_disk
from typing import Dict, List, Optional


def load_cl_depths(depths_file: str = 'cl_graph_depths.json') -> Dict[str, int]:
    """Load the CL ontology depths from the JSON file."""
    try:
        with open(depths_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Extract just the term_id -> depth mapping
        depths = {}
        for term_id, term_data in data.items():
            depths[term_id] = term_data.get('depth', 0)

        print(f"Loaded depths for {len(depths)} cell type terms")
        return depths

    except FileNotFoundError:
        print(f"Error: {depths_file} not found. Please run the NetworkX analyzer first.")
        raise
    except Exception as e:
        print(f"Error loading depths: {e}")
        raise


def add_depth_scores(dataset, cl_depths: Dict[str, int]):
    """Add depth scores to the dataset based on cell_type_ontology_term_id."""

    def add_depth(example):
        # Get the CL term ID for this example
        cl_term_id = example.get('cell_type_ontology_term_id', '')

        # Look up the depth, default to 0 if not found
        depth = cl_depths.get(cl_term_id, 0)

        # Add the depth to the example
        example['cl_depth'] = depth
        return example

    # Apply the depth addition to all examples
    dataset_with_depths = dataset.map(add_depth)

    return dataset_with_depths


def analyze_depth_distribution(dataset):
    """Analyze the distribution of depths in the dataset."""
    depths = [example['cl_depth'] for example in dataset]

    # Count occurrences of each depth
    depth_counts = {}
    for depth in depths:
        depth_counts[depth] = depth_counts.get(depth, 0) + 1

    print("\nDepth Distribution in Dataset:")
    print("Depth | Count | Percentage")
    print("-" * 30)

    total_examples = len(depths)
    for depth in sorted(depth_counts.keys()):
        count = depth_counts[depth]
        percentage = (count / total_examples) * 100
        print(f"{depth:5d} | {count:5d} | {percentage:6.1f}%")

    print(f"\nTotal examples: {total_examples}")
    print(f"Max depth: {max(depths)}")
    print(f"Average depth: {sum(depths) / total_examples:.2f}")


def create_curriculum_batches(dataset, batch_size: int = 1000, curriculum_order: str = 'ascending'):
    """
    Create curriculum learning batches ordered by CL depth.

    Args:
        dataset: Dataset with cl_depth field
        batch_size: Number of examples per batch
        curriculum_order: 'ascending' (easy to hard) or 'descending' (hard to easy)
    """

    # Sort dataset by depth
    reverse_order = (curriculum_order == 'descending')
    sorted_dataset = dataset.sort('cl_depth', reverse=reverse_order)

    # Create batches
    batches = []
    for i in range(0, len(sorted_dataset), batch_size):
        batch = sorted_dataset.select(range(i, min(i + batch_size, len(sorted_dataset))))
        batches.append(batch)

    print(f"\nCreated {len(batches)} curriculum batches")
    print(f"Order: {curriculum_order} (depth {sorted_dataset[0]['cl_depth']} → {sorted_dataset[-1]['cl_depth']})")

    return batches


def load_dataset_with_depths(hf_dataset,
                             curriculum_order: str = 'ascending',
                             batch_size: int = 1000,
                             depths_file: str = 'cl_graph_depths.json'):
    """
    Main function to add CL depth scores to existing dataset for curriculum learning.

    Args:
        hf_dataset: Pre-loaded Hugging Face dataset
        curriculum_order: 'ascending' for easy→hard, 'descending' for hard→easy
        batch_size: Size of curriculum batches
        depths_file: Path to the CL depths JSON file
    """

    print(f"Processing dataset with {len(hf_dataset)} examples")
    dataset = hf_dataset

    # Load CL depths
    cl_depths = load_cl_depths(depths_file)

    # Add depth scores
    print("Adding CL depth scores...")
    dataset_with_depths = add_depth_scores(dataset, cl_depths)

    # Analyze distribution
    analyze_depth_distribution(dataset_with_depths)

    # Create curriculum batches
    curriculum_batches = create_curriculum_batches(
        dataset_with_depths,
        batch_size=batch_size,
        curriculum_order=curriculum_order
    )

    # Show sample from first batch
    print(f"\nSample from first batch (depth-{curriculum_order}):")
    first_batch = curriculum_batches[0]
    for i in range(min(3, len(first_batch))):
        example = first_batch[i]
        print(f"  Depth {example['cl_depth']}: {example['cell_type']} ({example['cell_type_ontology_term_id']})")

    return dataset_with_depths, curriculum_batches


# Example usage for curriculum learning training loop
def curriculum_training_example(curriculum_batches):
    """
    Example training loop using curriculum learning batches.
    Replace with your actual training code.
    """
    print("\n" + "="*50)
    print("CURRICULUM TRAINING EXAMPLE")
    print("="*50)

    for batch_idx, batch in enumerate(curriculum_batches):
        batch_depths = [example['cl_depth'] for example in batch]
        min_depth = min(batch_depths)
        max_depth = max(batch_depths)

        print(f"\nBatch {batch_idx + 1}/{len(curriculum_batches)}")
        print(f"  Examples: {len(batch)}")
        print(f"  Depth range: {min_depth} - {max_depth}")

        # Your training code here
        # for example in batch:
        #       train_on_example(example)

        # Simulate training
        print(f"  Training on batch {batch_idx + 1}... (simulated)")



In [27]:
dataset_path = "/content/drive/MyDrive/cell2text_dataset_final/train"
depths_file_path = "cl_graph_depths.json" # Assuming depths file is also in Drive

print(f"Attempting to load dataset from: {dataset_path}")
hf_dataset = load_from_disk(dataset_path)

# Load dataset with depths and create curriculum batches
dataset_with_depths, curriculum_batches = load_dataset_with_depths(
    hf_dataset=hf_dataset,
    curriculum_order='ascending',  # Start with simple (low depth) cell types
    batch_size=1000,
    depths_file=depths_file_path # Pass the path to the depths file
)

    # Example curriculum training loop
if curriculum_batches:
    curriculum_training_example(curriculum_batches)

Attempting to load dataset from: /content/drive/MyDrive/cell2text_dataset_final/train
Processing dataset with 96499 examples
Loaded depths for 2922 cell type terms
Adding CL depth scores...


/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


Map:   0%|          | 0/96499 [00:00<?, ? examples/s]


Depth Distribution in Dataset:
Depth | Count | Percentage
------------------------------
    0 |   666 |    0.7%
    1 |   988 |    1.0%
    2 | 11549 |   12.0%
    3 | 15252 |   15.8%
    4 | 19475 |   20.2%
    5 | 17383 |   18.0%
    6 | 11891 |   12.3%
    7 |  5071 |    5.3%
    8 |  9344 |    9.7%
    9 |  2240 |    2.3%
   10 |  2631 |    2.7%
   11 |     9 |    0.0%

Total examples: 96499
Max depth: 11
Average depth: 4.80

Created 97 curriculum batches
Order: ascending (depth 0 → 11)

Sample from first batch (depth-ascending):
  Depth 0: CNS interneuron (CL:0000402)
  Depth 0: absorptive cell (CL:0000212)
  Depth 0: epithelial cell of alveolus of lung (CL:0010003)

CURRICULUM TRAINING EXAMPLE

Batch 1/97
  Examples: 1000
  Depth range: 0 - 1
  Training on batch 1... (simulated)

Batch 2/97
  Examples: 1000
  Depth range: 1 - 2
  Training on batch 2... (simulated)

Batch 3/97
  Examples: 1000
  Depth range: 2 - 2
  Training on batch 3... (simulated)

Batch 4/97
  Examples: 1000

In [30]:
import os

output_dataset_path = "/content/drive/MyDrive/cell2text_dataset_final/train_with_depths"
print(f"\nSaving dataset with depths to: {output_dataset_path}")
# Create the directory if it doesn't exist
os.makedirs(output_dataset_path, exist_ok=True)
dataset_with_depths.save_to_disk(output_dataset_path)
print("Dataset saved successfully!")


Saving dataset with depths to: /content/drive/MyDrive/cell2text_dataset_final/train_with_depths


Saving the dataset (0/2 shards):   0%|          | 0/96499 [00:00<?, ? examples/s]

Dataset saved successfully!
